# Experiment runner (DINOv2 + adapters)

This notebook is a **clean experiment template** for the histopathology OOD patch classification task.




# TO DO:

- 'tissue detection'/ artefacts : remove outliers like all black or all white (akub R. et al. A comprehensive solution to quality control problem in digital pathology. 2024.)

- Stain / Augmentations de couleur (H&E specific):(Khrystyna Faryna et al. Tailoring automated data augmentation to h&e-stained histopathology.) (David Tellez et al. Quantifying the effects of data augmentation and stain color normalization in convolutional neural networks for computational pathology. 2019). Augmentation de couleur avant normalisation

- augmentation geometriques (rotation, flip)

- instead of Dino : CTransPath, UNI, Virchow

- adapters: AdaptFormer ou LoRA/VeRA

## 0) Imports, paths, and utilities


This section initializes the notebook environment.

You will find:
- Python/PyTorch imports,
- path and project setup,
- helper utilities reused later.

In short: this is the **technical foundation** required before the rest of the pipeline.

In [ ]:
import os
import json
import time
import random
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any, Callable, Tuple, List

import h5py
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm.auto import tqdm

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    roc_curve,
    precision_recall_curve,
)

TRAIN_IMAGES_PATH = 'train.h5'
VAL_IMAGES_PATH = 'val.h5'
TEST_IMAGES_PATH = 'test.h5'

RUNS_DIR = 'runs'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch RNGs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str) -> None:
    """Create a directory if it does not exist."""
    os.makedirs(path, exist_ok=True)


def safe_json(x: Any) -> Any:
    """Best-effort JSON serialization for configs."""
    try:
        json.dumps(x)
        return x
    except TypeError:
        if hasattr(x, '__name__'):
            return x.__name__
        return str(x)


## 1) Config (edit only this section to define experiments)

You define experiments by creating a list of `RunConfig`.

### Flexibility
- Processing: resize + optional callable + optional sklearn-like transformer
- Backbone: choose any `torch.hub` DINOv2 name (or replace loader later)
- Adapter: any `nn.Module` class with signature `__init__(in_dim: int, **kwargs)`
- Head: any `nn.Module` class with signature `__init__(in_dim: int, **kwargs)` returning logits `(B, 1)`

### Output per run
- `runs/<run_name>/config.json`
- `runs/<run_name>/metrics.csv`
- `runs/<run_name>/checkpoints/best.pt` and `last.pt`
- Optional: `runs/<run_name>/predictions.csv`



This is the most important section to start experiments: it gathers the **experiment settings**.

You will find:
- dataclasses/config objects,
- training paths and runtime options,
- main hyperparameters.

In short: if you want to change a run, start with **this section**.

In [ ]:
@dataclass
class ProcessingConfig:
    """Image preprocessing configuration.

    Preprocessing order:
    1) Resize (torchvision)
    2) Optional `extra_transform` (callable on numpy array shaped (C, H, W))
    3) Optional sklearn-like transformer (object with `.transform(x)` or a callable)

    Notes
    - This template keeps shapes in (C, H, W).
    - If you want to use sklearn Pipelines, you typically wrap them so they accept/return
      numpy arrays with the same shape.
    """

    resize_hw: Tuple[int, int] = (98, 98)
    cast_float32: bool = True
    extra_transform: Optional[Callable[[np.ndarray], np.ndarray]] = None
    sklearn_transformer: Any = None


@dataclass
class ModuleSpec:
    """Generic module specification.

    The module class must accept `in_dim` as a keyword argument.

    Examples
    - Adapter: MLP adapter, prompt tokens, LoRA-like module, ...
    - Head: linear head, MLP head, cosine classifier, ...
    """

    enabled: bool = True
    module_cls: Optional[type] = None
    module_kwargs: Optional[Dict[str, Any]] = None


@dataclass
class ModelConfig:
    """Model configuration."""

    backbone_name: str = 'dinov2_vits14'
    adapter: ModuleSpec = ModuleSpec(enabled=True, module_cls=None, module_kwargs=None)
    head: ModuleSpec = ModuleSpec(enabled=True, module_cls=None, module_kwargs=None)


@dataclass
class EarlyStoppingConfig:
    """Early stopping configuration.

    Parameters
    - monitor: metric name to monitor on validation split.
      Supported: 'val_loss', 'val_auc', 'val_prauc', 'val_f1', 'val_accuracy'.
    - mode: 'min' for losses, 'max' for scores.
    - patience: number of epochs without improvement before stopping.
    - min_delta: minimum change to qualify as improvement.
    """

    monitor: str = 'val_loss'
    mode: str = 'min'
    patience: int = 10
    min_delta: float = 0.0


@dataclass
class TrainConfig:
    """Training configuration."""

    batch_size: int = 16
    lr: float = 1e-3
    weight_decay: float = 0.0
    num_epochs: int = 50

    early_stopping: EarlyStoppingConfig = EarlyStoppingConfig()

    use_center_balanced_batches: bool = False

    center_proportions: Optional[Dict[int, float]] = None

    num_workers: int = 0
    drop_last: bool = True


@dataclass
class RunConfig:
    """A single experiment configuration.

    `seeds` controls how many random seeds you run.
    Each seed generates its own sub-run folder:
    - runs/<run_name>/seed_<seed>/...

    Logging
    - Metrics are written incrementally to `metrics.csv` (one row per epoch and split).
    - Curves are saved to disk (ROC/PR) and referenced by path in the CSV.
    """

    run_name: str
    seeds: List[int]
    processing: ProcessingConfig
    model: ModelConfig
    train: TrainConfig

    do_predict_test: bool = False
    predict_threshold: float = 0.5


def save_run_config(run_cfg: RunConfig, run_dir: str) -> None:
    """Save config.json for reproducibility."""
    ensure_dir(run_dir)

    d = asdict(run_cfg)
    d['processing']['sklearn_transformer'] = safe_json(run_cfg.processing.sklearn_transformer)
    d['model']['adapter']['module_cls'] = safe_json(run_cfg.model.adapter.module_cls)
    d['model']['head']['module_cls'] = safe_json(run_cfg.model.head.module_cls)

    with open(os.path.join(run_dir, 'config.json'), 'w') as f:
        json.dump(d, f, indent=2)


## 2) Dataset and preprocessing

### What this section does

Here we prepare the **input data** for training and validation.

You will find:
- the `Dataset` class to read HDF5 data,
- image transforms/preprocessing,
- DataLoader creation.

In short: this is the **data pipeline** section (from files to model-ready batches).

In [ ]:
class H5BinaryDataset(Dataset):
    """H5 dataset for tumor/no-tumor classification.

    Expected H5 structure per key:
    - 'img': array-like tensor with shape (C, H, W)
    - 'label': scalar 0/1 for train/val
    - 'metadata': metadata array; metadata[0] is the center id (as in getting_started.ipynb)

    Returned samples:
    - mode in ('train','val'): (x, y, center_id)
    - mode == 'test': (x, id)
    """

    def __init__(self, h5_path: str, transform: Callable[[torch.Tensor], torch.Tensor], mode: str) -> None:
        super().__init__()
        self.h5_path = h5_path
        self.transform = transform
        self.mode = mode

        self._file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.image_ids = list(f.keys())

    def _get_file(self) -> h5py.File:
        if self._file is None:
            self._file = h5py.File(self.h5_path, 'r')
        return self._file

    def __len__(self) -> int:
        return len(self.image_ids)

    def __getitem__(self, idx: int):
        key = self.image_ids[idx]
        f = self._get_file()
        g = f[key]

        img = torch.tensor(np.array(g['img']))
        x = self.transform(img)

        if self.mode in ('train', 'val'):
            y = torch.tensor(np.array(g['label']).reshape(-1)[0]).float()
            meta = np.array(g.get('metadata')) if 'metadata' in g else np.array([np.nan])
            center_id = int(meta.reshape(-1)[0]) if meta.size else -1
            return x, y, center_id

        return x, int(key)


class SklearnLikeTransform:
    """Wrap a sklearn-like transformer into a callable."""

    def __init__(self, transformer: Any) -> None:
        self.transformer = transformer

    def __call__(self, x_np: np.ndarray) -> np.ndarray:
        if hasattr(self.transformer, 'transform'):
            return self.transformer.transform(x_np)
        if callable(self.transformer):
            return self.transformer(x_np)
        raise TypeError('sklearn_transformer must be callable or implement .transform(x).')


def build_preprocessing(cfg: ProcessingConfig) -> Callable[[torch.Tensor], torch.Tensor]:
    """Build a torch transform callable from ProcessingConfig."""

    resize = transforms.Resize(cfg.resize_hw)
    sklearn_transform = SklearnLikeTransform(cfg.sklearn_transformer) if cfg.sklearn_transformer is not None else None

    def _transform(x: torch.Tensor) -> torch.Tensor:
        y = resize(x)
        if cfg.cast_float32:
            y = y.float()

        if cfg.extra_transform is not None:
            y_np = y.detach().cpu().numpy().astype(np.float32)
            y_np = cfg.extra_transform(y_np)
            y = torch.tensor(y_np, dtype=torch.float32)

        if sklearn_transform is not None:
            y_np = y.detach().cpu().numpy().astype(np.float32)
            y_np = sklearn_transform(y_np)
            y = torch.tensor(y_np, dtype=torch.float32)

        return y

    return _transform


class CenterProportionalBatchSampler(Sampler[List[int]]):
    """Batch sampler enforcing (approximately) fixed center proportions per batch.

    This sampler yields lists of indices.

    It is designed for training only. It uses the center_id of each sample.

    Strategy:
    - Group indices by center
    - In each batch, draw a fixed number of samples from each center

    If exact proportions are not possible due to integer rounding, the remainder is
    assigned to the largest-probability centers.
    """

    def __init__(
        self,
        centers: List[int],
        batch_size: int,
        proportions: Dict[int, float],
        shuffle: bool = True,
        drop_last: bool = True,
        seed: int = 0,
    ) -> None:
        self.centers = np.array(centers, dtype=int)
        self.batch_size = int(batch_size)
        self.proportions = dict(proportions)
        self.shuffle = shuffle
        self.drop_last = drop_last
        self.rng = np.random.default_rng(seed)

        self.center_to_indices = {}
        for i, c in enumerate(self.centers.tolist()):
            self.center_to_indices.setdefault(int(c), []).append(i)

        for c in self.center_to_indices:
            self.center_to_indices[c] = np.array(self.center_to_indices[c], dtype=int)

        # Precompute per-batch counts
        probs = {int(k): float(v) for k, v in self.proportions.items()}
        # keep only centers that exist
        probs = {c: probs.get(c, 0.0) for c in self.center_to_indices.keys()}
        s = sum(probs.values())
        if s <= 0:
            # fallback: uniform
            probs = {c: 1.0 / len(probs) for c in probs}
        else:
            probs = {c: v / s for c, v in probs.items()}

        raw = {c: probs[c] * self.batch_size for c in probs}
        base = {c: int(np.floor(raw[c])) for c in raw}
        used = sum(base.values())
        remainder = self.batch_size - used
        # distribute remainder by largest fractional parts
        frac = sorted([(c, raw[c] - base[c]) for c in raw], key=lambda t: t[1], reverse=True)
        for c, _ in frac[:remainder]:
            base[c] += 1

        # Avoid zero counts for small centers if possible
        self.per_batch = base

        # Compute number of batches (limited by smallest center supply)
        limiting = []
        for c, cnt in self.per_batch.items():
            if cnt == 0:
                continue
            limiting.append(len(self.center_to_indices[c]) // cnt)
        self.num_batches = min(limiting) if limiting else 0

    def __len__(self) -> int:
        return self.num_batches

    def __iter__(self):
        # Create per-center streams
        streams = {}
        for c, idxs in self.center_to_indices.items():
            idxs = idxs.copy()
            if self.shuffle:
                self.rng.shuffle(idxs)
            streams[c] = idxs

        # Pointers per center
        ptr = {c: 0 for c in streams}

        for _ in range(self.num_batches):
            batch = []
            for c, cnt in self.per_batch.items():
                if cnt <= 0:
                    continue
                start = ptr[c]
                end = start + cnt
                batch.extend(streams[c][start:end].tolist())
                ptr[c] = end

            if len(batch) != self.batch_size:
                if self.drop_last:
                    continue
            if self.shuffle:
                self.rng.shuffle(batch)
            yield batch



## 3) Model building (frozen backbone + optional adapter + head)


### What this section does

This section builds the **final model** used for classification.

You will find:
- loading of the (frozen) backbone,
- optional adapter insertion,
- binary classification head,
- full network assembly.

In short: this is where we define the **architecture** to train.

In [ ]:
def load_frozen_backbone(backbone_name: str) -> nn.Module:
    '''Load a DINOv2 backbone from torch.hub and freeze it.'''
    backbone = torch.hub.load('facebookresearch/dinov2', backbone_name).to(DEVICE)
    backbone.eval()
    for p in backbone.parameters():
        p.requires_grad = False
    return backbone


class DefaultMLPAdapter(nn.Module):
    '''Default adapter: bottleneck MLP on embeddings.'''

    def __init__(self, in_dim: int, hidden_dim: int = 256, dropout: float = 0.1) -> None:
        super().__init__()
        self.down = nn.Linear(in_dim, hidden_dim)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.up = nn.Linear(hidden_dim, in_dim)
        self.alpha = nn.Parameter(torch.ones(1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.alpha * self.up(self.drop(self.act(self.down(x))))


class DefaultBinaryHead(nn.Module):
    '''Default head: linear layer producing logits (B, 1).'''

    def __init__(self, in_dim: int) -> None:
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


class FullModel(nn.Module):
    '''Monolithic model = backbone + adapter + head.'''

    def __init__(self, backbone: nn.Module, model_cfg: ModelConfig) -> None:
        super().__init__()
        self.backbone = backbone

        if not hasattr(backbone, 'num_features'):
            raise ValueError('Backbone is expected to expose `num_features`.')
        in_dim = int(backbone.num_features)

        self.adapter = None
        if model_cfg.adapter.enabled:
            cls = model_cfg.adapter.module_cls or DefaultMLPAdapter
            kwargs = model_cfg.adapter.module_kwargs or {}
            self.adapter = cls(in_dim=in_dim, **kwargs)

        if not model_cfg.head.enabled:
            raise ValueError('Head must be enabled for classification.')
        head_cls = model_cfg.head.module_cls or DefaultBinaryHead
        head_kwargs = model_cfg.head.module_kwargs or {}
        self.head = head_cls(in_dim=in_dim, **head_kwargs)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.backbone(x)
        if self.adapter is not None:
            z = self.adapter(z)
        logits = self.head(z)
        return logits


def trainable_parameters(model: nn.Module) -> List[nn.Parameter]:
    '''Return parameters that will be optimized (adapter + head).'''
    return [p for p in model.parameters() if p.requires_grad]


## 4) Training loop + logging + checkpoints


Here we define **how the model learns** and how performance is tracked.

You will find:
- training/validation loops,
- metrics and logs (CSV, console output),
- checkpoint saving.

In short: this is the **training and monitoring** section.

In [ ]:
class CSVLogger:
    """Append-only CSV logger for epoch-level metrics.

    The logger writes one row per (epoch, split).
    Extra metadata (config-like fields) are repeated on each row to make
    filtering/plotting in pandas easy.
    """

    def __init__(self, csv_path: str, extra_fieldnames: List[str], metric_fieldnames: List[str]) -> None:
        self.csv_path = csv_path
        self.extra_fieldnames = list(extra_fieldnames)
        self.metric_fieldnames = list(metric_fieldnames)
        ensure_dir(os.path.dirname(self.csv_path))

        if not os.path.exists(self.csv_path):
            header = ['run_name', 'epoch', 'split'] + self.metric_fieldnames + self.extra_fieldnames
            pd.DataFrame(columns=header).to_csv(self.csv_path, index=False)

    def log(self, run_name: str, epoch: int, split: str, metrics: Dict[str, Any], extra: Dict[str, Any]) -> None:
        row = {
            'run_name': run_name,
            'epoch': int(epoch),
            'split': str(split),
        }

        for k in self.metric_fieldnames:
            row[k] = safe_json(metrics.get(k))

        for k in self.extra_fieldnames:
            row[k] = safe_json(extra.get(k))

        pd.DataFrame([row]).to_csv(self.csv_path, mode='a', header=False, index=False)


def _safe_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Compute ROC AUC, returning NaN if undefined."""
    try:
        return float(roc_auc_score(y_true, y_score))
    except Exception:
        return float('nan')


def _safe_prauc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Compute PR AUC (Average Precision), returning NaN if undefined."""
    try:
        return float(average_precision_score(y_true, y_score))
    except Exception:
        return float('nan')


def compute_binary_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = 0.5,
) -> Dict[str, float]:
    """Compute binary classification metrics.

    Parameters
    - y_true: shape (N,), values in {0,1}
    - y_prob: shape (N,), probabilities in [0,1]
    - threshold: probability threshold for hard predictions

    Returns
    - dict with accuracy, f1, auc, prauc
    """

    y_true = y_true.astype(int).reshape(-1)
    y_prob = y_prob.astype(float).reshape(-1)
    y_pred = (y_prob >= threshold).astype(int)

    out = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'auc': _safe_auc(y_true, y_prob),
        'prauc': _safe_prauc(y_true, y_prob),
    }
    return out


def save_curves_npz(out_path: str, y_true: np.ndarray, y_prob: np.ndarray) -> None:
    """Save ROC and PR curves (numpy arrays) to a .npz file."""

    y_true = y_true.astype(int).reshape(-1)
    y_prob = y_prob.astype(float).reshape(-1)

    try:
        fpr, tpr, roc_thr = roc_curve(y_true, y_prob)
    except Exception:
        fpr, tpr, roc_thr = np.array([]), np.array([]), np.array([])

    try:
        prec, rec, pr_thr = precision_recall_curve(y_true, y_prob)
    except Exception:
        prec, rec, pr_thr = np.array([]), np.array([]), np.array([])

    ensure_dir(os.path.dirname(out_path))
    np.savez(out_path, fpr=fpr, tpr=tpr, roc_thresholds=roc_thr, precision=prec, recall=rec, pr_thresholds=pr_thr)


class EarlyStopping:
    """Metric-based early stopping."""

    def __init__(self, cfg: EarlyStoppingConfig) -> None:
        self.cfg = cfg
        self.best = None
        self.best_epoch = -1
        self.num_bad = 0

    def _is_better(self, value: float) -> bool:
        if self.best is None:
            return True
        if self.cfg.mode == 'min':
            return value < (self.best - self.cfg.min_delta)
        return value > (self.best + self.cfg.min_delta)

    def step(self, epoch: int, metrics: Dict[str, Any]) -> bool:
        """Return True if training should stop."""

        v = metrics.get(self.cfg.monitor)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            self.num_bad += 1
            return self.num_bad >= self.cfg.patience

        v = float(v)
        if self._is_better(v):
            self.best = v
            self.best_epoch = epoch
            self.num_bad = 0
        else:
            self.num_bad += 1

        return self.num_bad >= self.cfg.patience


class Trainer:
    """Trainer that logs global + per-center metrics and supports early stopping."""

    def __init__(
        self,
        model: nn.Module,
        train_cfg: TrainConfig,
        optimizer: optim.Optimizer,
        criterion: nn.Module,
        logger: CSVLogger,
        run_name: str,
        ckpt_dir: str,
        curves_dir: str,
        centers_in_train: List[int],
        threshold: float = 0.5,
    ) -> None:
        self.model = model
        self.train_cfg = train_cfg
        self.optimizer = optimizer
        self.criterion = criterion
        self.logger = logger
        self.run_name = run_name
        self.ckpt_dir = ckpt_dir
        self.curves_dir = curves_dir
        self.centers_in_train = sorted(list(set(int(c) for c in centers_in_train if int(c) >= 0)))
        self.threshold = float(threshold)

        ensure_dir(self.ckpt_dir)
        ensure_dir(self.curves_dir)

    def _forward_probs(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.model(x)
        return torch.sigmoid(logits).view(-1)

    def _epoch_pass(self, loader: DataLoader, train: bool) -> Dict[str, Any]:
        if train:
            self.model.train()
        else:
            self.model.eval()

        losses = []
        y_true_all, y_prob_all, centers_all = [], [], []

        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            it = tqdm(loader, leave=False, desc='train' if train else 'eval')
            for batch in it:
                if len(batch) == 3:
                    x, y, center = batch
                else:
                    x, y = batch
                    center = torch.full((x.shape[0],), -1, dtype=torch.long)

                x = x.to(DEVICE)
                y = y.to(DEVICE).view(-1).float()

                if train:
                    self.optimizer.zero_grad(set_to_none=True)

                logits = self.model(x).view(-1)
                prob = torch.sigmoid(logits)
                loss = self.criterion(logits.view(-1, 1), y.view(-1, 1))

                if train:
                    loss.backward()
                    self.optimizer.step()

                losses.append(float(loss.detach().item()))

                y_true_all.append(y.detach().cpu().numpy())
                y_prob_all.append(prob.detach().cpu().numpy())
                centers_all.append(center.detach().cpu().numpy().astype(int).reshape(-1))

        y_true = np.concatenate(y_true_all, axis=0)
        y_prob = np.concatenate(y_prob_all, axis=0)
        centers = np.concatenate(centers_all, axis=0)

        metrics_global = compute_binary_metrics(y_true, y_prob, threshold=self.threshold)
        out = {
            'loss': float(np.mean(losses)) if losses else float('nan'),
            'accuracy': metrics_global['accuracy'],
            'f1': metrics_global['f1'],
            'auc': metrics_global['auc'],
            'prauc': metrics_global['prauc'],
        }

        for c in self.centers_in_train:
            m = (centers == int(c))
            if m.sum() == 0:
                out[f'center_{c}_accuracy'] = float('nan')
                out[f'center_{c}_f1'] = float('nan')
                out[f'center_{c}_auc'] = float('nan')
                out[f'center_{c}_prauc'] = float('nan')
            else:
                cm = compute_binary_metrics(y_true[m], y_prob[m], threshold=self.threshold)
                out[f'center_{c}_accuracy'] = cm['accuracy']
                out[f'center_{c}_f1'] = cm['f1']
                out[f'center_{c}_auc'] = cm['auc']
                out[f'center_{c}_prauc'] = cm['prauc']

        return out, y_true, y_prob

    def fit(self, train_loader: DataLoader, val_loader: DataLoader, extra: Dict[str, Any]) -> Dict[str, Any]:
        es = EarlyStopping(self.train_cfg.early_stopping)

        best_epoch = -1
        history = []

        for epoch in range(self.train_cfg.num_epochs):
            tr_metrics, _, _ = self._epoch_pass(train_loader, train=True)
            va_metrics, va_y, va_p = self._epoch_pass(val_loader, train=False)

            curves_path = os.path.join(self.curves_dir, f'val_epoch_{epoch:03d}.npz')
            save_curves_npz(curves_path, va_y, va_p)

            log_train = dict(tr_metrics)
            log_val = dict(va_metrics)

            log_train['rocpr_path'] = ''
            log_val['rocpr_path'] = curves_path

            self.logger.log(self.run_name, epoch, 'train', log_train, extra)
            self.logger.log(self.run_name, epoch, 'val', log_val, extra)

            torch.save({'model_state': self.model.state_dict()}, os.path.join(self.ckpt_dir, 'last.pt'))

            monitored = {
                'val_loss': log_val.get('loss'),
                'val_accuracy': log_val.get('accuracy'),
                'val_f1': log_val.get('f1'),
                'val_auc': log_val.get('auc'),
                'val_prauc': log_val.get('prauc'),
            }

            stop = es.step(epoch, monitored)
            if es.best_epoch == epoch:
                best_epoch = epoch
                torch.save({'model_state': self.model.state_dict()}, os.path.join(self.ckpt_dir, 'best.pt'))

            history.append({'epoch': epoch, 'train': tr_metrics, 'val': va_metrics, 'monitor': monitored})

            if stop:
                break

        best_val = None
        if es.best is not None:
            best_val = float(es.best)

        return {
            'best_epoch': int(best_epoch),
            'early_stopping_monitor': self.train_cfg.early_stopping.monitor,
            'best_monitor_value': best_val,
            'history': history,
        }


## 5) Optional: predict on test.h5


Optional section to **generate predictions on `test.h5`** from a trained model.

You will find:
- the prediction function,
- expected output format,
- inference logic (no training).

In short: useful when moving from a trained model to **usable predictions**.

In [ ]:
def predict_test(run_dir: str, model: nn.Module, transform: Callable[[torch.Tensor], torch.Tensor], threshold: float) -> pd.DataFrame:
    '''Run inference on test.h5 and write predictions.csv.'''

    test_ds = H5BinaryDataset(TEST_IMAGES_PATH, transform=transform, mode='test')
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

    model.eval()
    rows = []
    with torch.no_grad():
        for x, img_id in tqdm(test_loader, leave=False, desc='predict'):
            x = x.to(DEVICE)
            logits = model(x)
            prob = torch.sigmoid(logits).item()
            rows.append({'ID': int(img_id.item()), 'Pred': int(prob > threshold)})

    out = pd.DataFrame(rows).set_index('ID')
    out_path = os.path.join(run_dir, 'predictions.csv')
    out.to_csv(out_path)
    return out


## 6) Experiment runner


This section orchestrates the **full pipeline for one experiment**: setup, training, evaluation, and saving artifacts.

You will find:
- the execution flow that chains steps in the right order,
- output directory management for each run,
- coordination utilities (loading metadata, execution, recap).

In short: this is the notebook's **orchestrator**.

In [ ]:
def _load_centers_from_h5(h5_path: str, image_ids: List[str]) -> List[int]:
    """Load center ids from the H5 metadata (metadata[0])."""

    centers = []
    with h5py.File(h5_path, 'r') as f:
        for k in image_ids:
            g = f[k]
            if 'metadata' not in g:
                centers.append(-1)
                continue
            meta = np.array(g.get('metadata')).reshape(-1)
            centers.append(int(meta[0]) if meta.size else -1)
    return centers


def _estimate_center_proportions(centers: List[int]) -> Dict[int, float]:
    """Estimate center proportions from a list of center ids."""

    c = np.array([int(x) for x in centers if int(x) >= 0], dtype=int)
    if c.size == 0:
        return {}
    uniq, cnt = np.unique(c, return_counts=True)
    p = cnt / cnt.sum()
    return {int(u): float(v) for u, v in zip(uniq, p)}


def run_experiment(run_cfg: RunConfig) -> Dict[str, Any]:
    """Run one experiment across one or more random seeds.

    Folder structure
    - runs/<run_name>/config.json
    - runs/<run_name>/seed_<seed>/metrics.csv
    - runs/<run_name>/seed_<seed>/checkpoints/{best,last}.pt
    - runs/<run_name>/seed_<seed>/curves/val_epoch_XXX.npz
    """

    base_run_dir = os.path.join(RUNS_DIR, run_cfg.run_name)
    ensure_dir(base_run_dir)
    save_run_config(run_cfg, base_run_dir)

    transform = build_preprocessing(run_cfg.processing)

    seed_results = []

    for seed in run_cfg.seeds:
        seed_everything(int(seed))

        run_dir = os.path.join(base_run_dir, f'seed_{int(seed)}')
        ckpt_dir = os.path.join(run_dir, 'checkpoints')
        curves_dir = os.path.join(run_dir, 'curves')
        metrics_path = os.path.join(run_dir, 'metrics.csv')

        ensure_dir(run_dir)
        ensure_dir(ckpt_dir)
        ensure_dir(curves_dir)

        train_ds = H5BinaryDataset(TRAIN_IMAGES_PATH, transform=transform, mode='train')
        val_ds = H5BinaryDataset(VAL_IMAGES_PATH, transform=transform, mode='val')

        train_centers = _load_centers_from_h5(TRAIN_IMAGES_PATH, train_ds.image_ids)
        centers_in_train = sorted(list(set(int(c) for c in train_centers if int(c) >= 0)))

        if run_cfg.train.use_center_balanced_batches:
            proportions = run_cfg.train.center_proportions
            if proportions is None:
                proportions = _estimate_center_proportions(train_centers)

            batch_sampler = CenterProportionalBatchSampler(
                centers=train_centers,
                batch_size=run_cfg.train.batch_size,
                proportions=proportions,
                shuffle=True,
                drop_last=run_cfg.train.drop_last,
                seed=int(seed),
            )
            train_loader = DataLoader(
                train_ds,
                batch_sampler=batch_sampler,
                num_workers=run_cfg.train.num_workers,
            )
        else:
            train_loader = DataLoader(
                train_ds,
                batch_size=run_cfg.train.batch_size,
                shuffle=True,
                num_workers=run_cfg.train.num_workers,
                drop_last=run_cfg.train.drop_last,
            )

        val_loader = DataLoader(
            val_ds,
            batch_size=run_cfg.train.batch_size,
            shuffle=False,
            num_workers=run_cfg.train.num_workers,
            drop_last=False,
        )

        backbone = load_frozen_backbone(run_cfg.model.backbone_name)
        model = FullModel(backbone, run_cfg.model).to(DEVICE)

        params = trainable_parameters(model)
        if len(params) == 0:
            raise RuntimeError('No trainable parameters found. Make sure adapter/head are enabled.')

        optimizer = optim.Adam(params, lr=run_cfg.train.lr, weight_decay=run_cfg.train.weight_decay)
        criterion = nn.BCEWithLogitsLoss()

        extra = {
            'seed': int(seed),
            'resize_hw': f'{run_cfg.processing.resize_hw[0]}x{run_cfg.processing.resize_hw[1]}',
            'adapter_enabled': bool(run_cfg.model.adapter.enabled),
            'adapter_cls': safe_json(run_cfg.model.adapter.module_cls),
            'head_cls': safe_json(run_cfg.model.head.module_cls),
            'backbone': run_cfg.model.backbone_name,
            'use_center_balanced_batches': bool(run_cfg.train.use_center_balanced_batches),
            'center_proportions': safe_json(run_cfg.train.center_proportions),
            'early_stopping_monitor': run_cfg.train.early_stopping.monitor,
            'early_stopping_mode': run_cfg.train.early_stopping.mode,
            'early_stopping_patience': run_cfg.train.early_stopping.patience,
        }

        metric_fields = ['loss', 'accuracy', 'f1', 'auc', 'prauc', 'rocpr_path']
        for c in centers_in_train:
            metric_fields += [
                f'center_{c}_accuracy', f'center_{c}_f1', f'center_{c}_auc', f'center_{c}_prauc'
            ]

        logger = CSVLogger(metrics_path, extra_fieldnames=list(extra.keys()), metric_fieldnames=metric_fields)

        trainer = Trainer(
            model=model,
            train_cfg=run_cfg.train,
            optimizer=optimizer,
            criterion=criterion,
            logger=logger,
            run_name=run_cfg.run_name,
            ckpt_dir=ckpt_dir,
            curves_dir=curves_dir,
            centers_in_train=centers_in_train,
            threshold=run_cfg.predict_threshold,
        )

        t0 = time.time()
        fit = trainer.fit(train_loader, val_loader, extra=extra)
        t1 = time.time()

        fit['seed'] = int(seed)
        fit['time_sec'] = float(t1 - t0)
        seed_results.append(fit)

        if run_cfg.do_predict_test:
            ckpt = torch.load(os.path.join(ckpt_dir, 'best.pt'), map_location='cpu')
            model.load_state_dict(ckpt['model_state'])
            predict_test(run_dir, model, transform, threshold=run_cfg.predict_threshold)

    return {
        'run_name': run_cfg.run_name,
        'seeds': list(run_cfg.seeds),
        'seed_results': seed_results,
    }


## 7) Define runs and launch training


Here we **define which experiments to run** (the `RUNS` list) and then start training.

You will mainly find:
- configuration variants to compare (hyperparameters, seed, etc.),
- the final call that launches each run,
- the practical entry point for quickly testing ideas.

In short: this is the **"choose runs and launch"** section.

In [ ]:
RUNS: List[RunConfig] = [
    RunConfig(
        run_name='example_adapter_mlp',
        seeds=[0],
        processing=ProcessingConfig(resize_hw=(98, 98)),
        model=ModelConfig(
            backbone_name='dinov2_vits14',
            adapter=ModuleSpec(
                enabled=True,
                module_cls=DefaultMLPAdapter,
                module_kwargs={'hidden_dim': 256, 'dropout': 0.1},
            ),
            head=ModuleSpec(
                enabled=True,
                module_cls=DefaultBinaryHead,
                module_kwargs={},
            ),
        ),
        train=TrainConfig(
            batch_size=16,
            lr=1e-3,
            num_epochs=10,
            early_stopping=EarlyStoppingConfig(monitor='val_loss', mode='min', patience=3),
            use_center_balanced_batches=True,
            center_proportions=None,
        ),
        do_predict_test=False,
    ),
]

results = []
for rc in RUNS:
    print(f"\n=== Starting run: {rc.run_name} ===")
    out = run_experiment(rc)
    results.append(out)

rows = []
for r in results:
    for sr in r['seed_results']:
        rows.append({
            'run_name': r['run_name'],
            'seed': sr['seed'],
            'best_epoch': sr.get('best_epoch'),
            'monitor': sr.get('early_stopping_monitor'),
            'best_monitor_value': sr.get('best_monitor_value'),
            'time_sec': sr.get('time_sec'),
        })

pd.DataFrame(rows).sort_values(['run_name', 'seed'])
